In [38]:
import pandas as pd

In [39]:
course = pd.read_csv('../data-csv/course.csv', parse_dates=['date_started'])

In [40]:
course['max_score'].dtype # для справочки если есть nan среди int то тип колонки - float

dtype('float64')

In [41]:
user_course_progress = pd.read_csv('../data-csv/user_course_progress.csv')

#### Относительным прогрессом будем называть отношение суммарного числа баллов, которые ученик набрал за курс (колонка progress в таблице user_course_progress), к максимальному количеству баллов, которые ученик в принципе мог набрать за данный курс (колонка max_progress в таблице course).

In [42]:
user_course_progress

,user_id,course_id,progress
0,392,1,31.68
1,217,1,97.53
2,843,1,81.50
3,417,1,15.92
4,605,1,103.31
...,...,...,...
1706,898,10,8.01
1707,201,10,0.71
1708,334,10,1.90
1709,112,10,2.16


## На курсах какого предмета больше всего учеников, чей относительный прогресс не меньше 0.8?

In [43]:
course_progress = course \
    .merge(user_course_progress, left_on='id', right_on='course_id', suffixes=('_x','_y'), how='inner') 
    

In [44]:
len(course_progress[course_progress['subject'] == 'Химия'])

678

In [45]:
course_progress = course_progress.dropna(subset=['max_score'])

In [46]:
course_progress

,id,subject,date_started,max_score,user_id,course_id,progress
0,1,Химия,2023-04-04,108.0,392,1,31.68
1,1,Химия,2023-04-04,108.0,217,1,97.53
2,1,Химия,2023-04-04,108.0,843,1,81.50
3,1,Химия,2023-04-04,108.0,417,1,15.92
4,1,Химия,2023-04-04,108.0,605,1,103.31
...,...,...,...,...,...,...,...
1706,10,Математика,2023-02-18,104.0,898,10,8.01
1707,10,Математика,2023-02-18,104.0,201,10,0.71
1708,10,Математика,2023-02-18,104.0,334,10,1.90
1709,10,Математика,2023-02-18,104.0,112,10,2.16


In [47]:
course_progress = course_progress.drop(columns=['id','date_started']) #убираем ненужные колонки

In [48]:
course_progress

,subject,max_score,user_id,course_id,progress
0,Химия,108.0,392,1,31.68
1,Химия,108.0,217,1,97.53
2,Химия,108.0,843,1,81.50
3,Химия,108.0,417,1,15.92
4,Химия,108.0,605,1,103.31
...,...,...,...,...,...
1706,Математика,104.0,898,10,8.01
1707,Математика,104.0,201,10,0.71
1708,Математика,104.0,334,10,1.90
1709,Математика,104.0,112,10,2.16


In [49]:
# Каждый предмет содержит в себе несколько курсов, которые могут проходить ребята ОДНОВРЕМЕННО
course_progress['subject'].unique()

<StringArray>
['Химия', 'Физика', 'Математика']
Length: 3, dtype: str

In [57]:
course_progress['relative_progress'] = (course_progress['progress'] / course_progress['max_score'])


In [58]:
course_progress['relative_progress'].mean().round(2)

np.float64(0.35)

In [59]:
course_mt_80 = course_progress[course_progress['relative_progress'] >= 0.8]

In [60]:
len(course_mt_80['user_id'])

159

In [54]:
course_mt_80[course_mt_80['subject']=='Химия']

,subject,max_score,user_id,course_id,progress,relative_progress
1,Химия,108.0,217,1,97.53,0.90
4,Химия,108.0,605,1,103.31,0.96
25,Химия,108.0,720,1,105.11,0.97
28,Химия,108.0,679,1,97.80,0.91
29,Химия,108.0,364,1,92.21,0.85
...,...,...,...,...,...,...
1470,Химия,120.0,101,9,115.36,0.96
1479,Химия,120.0,27,9,115.83,0.97
1491,Химия,120.0,838,9,118.97,0.99
1504,Химия,120.0,520,9,108.40,0.90


In [61]:
course_mt_80 = course_mt_80[['subject', 'user_id','course_id']].sort_values('user_id')

In [62]:
course_mt_80.groupby(['subject']).agg({'user_id': 'count'}).sort_values('user_id', ascending=False)

,user_id
subject,
Химия,105
Математика,30
Физика,24
